In [ ]:
from google.colab import drive
import os
# Google Drive 마운트
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import json
import re
import nltk
from collections import Counter # TTR 계산을 위해 필요

# --- VADER Sentiment Analyzer 설정 ---
# VADER는 외부 라이브러리 및 데이터를 필요로 하므로 try-except 블록으로 처리합니다.
try:
    # nltk 다운로드는 Colab 환경에서 자동으로 처리됩니다.
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    nltk.download("vader_lexicon", quiet=True)
    sia = SentimentIntensityAnalyzer()
    print("✅ VADER Sentiment Analyzer 초기화 완료.")
except Exception as e:
    print(f"⚠️ VADER 초기화 실패: {e}")
    sia = None

# 1. melon_monthly_chart를 데이터 프레임으로 만들기
try:
    df = pd.read_csv('melon_monthly_chart.csv')
    print("✅ 'melon_monthly_chart.csv' 파일 로드 완료.")
except FileNotFoundError:
    print("🚨 오류: 'melon_monthly_chart.csv' 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인해주세요.")
    df = None

# SentiWord_info.json을 불러와 한국어 감성 사전 생성
korean_senti_dict = {}
try:
    with open('SentiWord_info.json', 'r', encoding='utf-8') as f:
        senti_data = json.load(f)
    korean_senti_dict = {item['word']: int(item['polarity']) for item in senti_data}
    print("✅ 'SentiWord_info.json' (한국어 감성 사전) 로드 완료.")
except FileNotFoundError:
    print("🚨 오류: 'SentiWord_info.json' 파일을 찾을 수 없습니다. 한국어 감성 분석은 건너뜜니다.")
except Exception as e:
    print(f"🚨 오류: 'SentiWord_info.json' 처리 중 예외 발생: {e}")

# 데이터프레임이 로드된 경우에만 분석 진행
if df is not None:
    # 2. 한국어 가사 감성 분석 함수 정의 및 적용
    def get_korean_polarity(lyrics):
        """SentiWord_info 사전을 사용하여 한국어 가사의 평균 감성 극성(polarity)을 계산합니다."""
        if not korean_senti_dict or not isinstance(lyrics, str) or pd.isna(lyrics):
            return None

        # 구두점 제거 및 띄어쓰기 단위로 분석
        cleaned_lyrics = re.sub(r'[^\w\s]', '', lyrics)
        words = cleaned_lyrics.split()
        polarities = []
        for word in words:
            if word in korean_senti_dict:
                polarities.append(korean_senti_dict[word])

        # 감성 단어가 발견되지 않으면 0.0을 반환
        return sum(polarities) / len(polarities) if polarities else 0.0

    df['Korean_Avg_Polarity'] = df['lyrics'].apply(get_korean_polarity)
    print("✅ 한국어 감성 극성 계산 완료.")


    # 3. 영어 가사 감성 분석 함수 정의 및 적용 (VADER 사용)
    def get_english_polarity(lyrics):
        """VADER Sentiment Analyzer를 사용하여 영어 가사의 Compound Score를 계산합니다."""
        # VADER 초기화 실패 시 (위의 try-except 블록) None 반환
        if sia is None or not isinstance(lyrics, str) or pd.isna(lyrics) or not lyrics.strip():
            return None

        # VADER는 전체 텍스트에 대한 감성 점수(Compound Score)를 반환합니다.
        score = sia.polarity_scores(lyrics)['compound']
        return score

    df['English_Avg_Polarity'] = df['lyrics'].apply(get_english_polarity)
    print("✅ 영어 감성 극성 계산 완료.")


    # 4. 한국어 단어 포함 여부로 한국 노래 판별 함수 정의 및 적용
    def is_korean_song(lyrics):
        """가사에 한국어(한글)이 포함되어 있는지 확인합니다."""
        if not isinstance(lyrics, str) or pd.isna(lyrics):
            return False
        # 정규 표현식으로 한글 유무 확인
        return bool(re.search(r'[가-힣]', lyrics))

    df['Is_Korean_Song'] = df['lyrics'].apply(is_korean_song)
    print("✅ 한국 노래 판별 완료.")


    # 5. Lexical Diversity (Type-Token Ratio, TTR)를 계산하는 함수 정의 및 적용
    def calculate_ttr(lyrics):
        """
        가사의 어휘 다양성(Type-Token Ratio)을 계산합니다.
        TTR = (고유한 단어 수) / (전체 단어 수)
        """
        if not isinstance(lyrics, str) or pd.isna(lyrics):
            return None

        # 1. 소문자 변환 및 구두점/특수 문자 제거
        cleaned_lyrics = re.sub(r'[^\w\s]', '', lyrics.lower())

        # 2. 띄어쓰기 단위로 토큰화
        tokens = cleaned_lyrics.split()

        if not tokens:
            return 0.0

        # 3. TTR 계산
        num_tokens = len(tokens)
        num_types = len(set(tokens)) # 고유한 단어의 수

        return num_types / num_tokens

    df['Lexical_Diversity'] = df['lyrics'].apply(calculate_ttr)
    print("✅ 어휘 다양성 (TTR) 계산 완료.")


    # 6. 두 감성 극성 평균 계산
    # .mean(axis=1)은 결측값(None/NaN)을 제외하고 평균을 계산합니다.
    df['Combined_Avg_Polarity'] = df[['Korean_Avg_Polarity', 'English_Avg_Polarity']].mean(axis=1)
    print("✅ 통합 평균 감성 극성 계산 완료.")


    # 7. Google Drive에 CSV 파일로 저장
    output_path = '/content/drive/MyDrive/final_melon_chart_analysis.csv'
    try:
        if not os.path.exists('/content/drive/MyDrive'):
             print("\n❗ Google Drive가 마운트되지 않았습니다. Drive 마운트 코드를 먼저 실행해주세요.")
        else:
            # 최종 데이터프레임을 CSV로 저장 (한글 깨짐 방지를 위해 'utf-8-sig' 사용)
            df.to_csv(output_path, index=False, encoding='utf-8-sig')

            print("\n" + "="*70)
            print("🎉 데이터 처리 및 저장 완료!")
            print(f"💾 CSV 파일이 Google Drive에 성공적으로 저장되었습니다.")
            print(f"저장 경로: {output_path}")
            print("="*70)

    except Exception as e:
        print(f"\n❌ Google Drive 저장 중 오류 발생: {e}")

    print("\n👇 최종 데이터프레임 미리보기 (상위 5개 행):")
    print(df.head())
    print("\n👇 최종 데이터프레임 정보:")
    df.info()

✅ VADER Sentiment Analyzer 초기화 완료.
✅ 'melon_monthly_chart.csv' 파일 로드 완료.
✅ 'SentiWord_info.json' (한국어 감성 사전) 로드 완료.
✅ 한국어 감성 극성 계산 완료.
✅ 영어 감성 극성 계산 완료.
✅ 한국 노래 판별 완료.
✅ 어휘 다양성 (TTR) 계산 완료.
✅ 통합 평균 감성 극성 계산 완료.

🎉 데이터 처리 및 저장 완료!
💾 CSV 파일이 Google Drive에 성공적으로 저장되었습니다.
저장 경로: /content/drive/MyDrive/final_melon_chart_analysis.csv

👇 최종 데이터프레임 미리보기 (상위 5개 행):
        song_name                    artist genre  \
0  OHAYO MY NIGHT               디핵 (D-Hack)  랩/힙합   
1     Off My Face             Justin Bieber   POP   
2       Shut Down                 BLACKPINK  랩/힙합   
3           추억때문에                        유림   발라드   
4       FOREVER 1  소녀시대 (GIRLS' GENERATION)    댄스   

                                              lyrics  year  month  rank  \
0  너를 사랑하고 있어\n너를 사랑하고 있어\n자기야 날 사랑해주면 안 될까\n말처럼 ...  2022     11    50   
1  One touch and you got me stoned\nHigher than I...  2022     11    50   
2  Blackpink in your area\nBlackpink in your area...  2022     12    16   
3  안녕이라는 말은 하지 마\n마음이 